In [1]:
import pandas as pd
import numpy as np
from scipy.signal import spectrogram
import holoviews as hv 
import panel as pn
from scipy.io import wavfile
hv.extension("bokeh", logo=False)

# Маркировка

In [2]:
def update_playhead(x,y,t):
    if x is None:
        return hv.VLine(t)
    else:
        audio.time = x
        return hv.VLine(x)

In [3]:
"""
- - остаётся plain
! - шум
+ - размечено
sample_numchecks: 
1-, 
2-, 
3-, 
4!, 
5!, 
6+, 
7!,
""" 

'\n- - остаётся plain\n! - шум\n+ - размечено\nsample_numchecks: \n1-, \n2-, \n3-, \n4!, \n5!, \n6+, \n7!,\n'

In [4]:
sample_num = 6
a_s_num = 1

In [5]:
filters = [
    ('sample_num', '=', sample_num),
    ('a_s_num', '=', a_s_num),
]
df_percussive = pd.read_parquet('audiosets_percussive.parquet', columns=['percussive'], filters=filters)
df_percussive.head()

,percussive
169472,-0.003638
169473,-0.003529
169474,-0.001671
169475,-0.004711
169476,-0.012969


In [6]:
df_percussive.shape

(56833, 1)

In [7]:
sr = 22050
audio_data = df_percussive.percussive.to_numpy()
f, t, sxx = spectrogram(audio_data, sr)
spec_gram = hv.Image((t, f, np.log2(sxx)), ["Time (s)", "Frequency (hz)"]).opts(width=1280, colorbar=True)
audio = pn.pane.Audio(audio_data, sample_rate=sr, name='Audio', throttle=100)
tap_stream = hv.streams.SingleTap(transient=True)
time_play_stream = hv.streams.Params(parameters=[audio.param.time], rename={'time': 't'})
dmap_time = hv.DynamicMap(update_playhead, streams=[time_play_stream, tap_stream])
out = pn.Column( audio, 
               (spec_gram * dmap_time))
out

Column
    [0] Audio(ndarray, sample_rate=22050, throttle=100)
    [1] HoloViews(DynamicMap, height=300, sizing_mode='fixed', width=1280)

In [8]:
df_main = pd.read_parquet('audiosets_main_20_04.parquet')
df_main.head()

,time_mark,sample_num,a_s_duration,a_s_sr,a_s_num,mark
0,0.000000,0,41.052517,22050,1,plain
1,0.000045,0,41.052517,22050,1,plain
2,0.000091,0,41.052517,22050,1,plain
3,0.000136,0,41.052517,22050,1,plain
4,0.000181,0,41.052517,22050,1,plain


In [9]:
slice_start = df_main.loc[
    (df_main.sample_num == sample_num) & 
    (df_main.a_s_num == a_s_num)
].time_mark.min()

In [10]:
start, stop = slice_start + 0.97, slice_start + 1.20

In [12]:
df_main.loc[
    (df_main.sample_num == sample_num) & 
    (df_main.a_s_num == a_s_num) &
    (df_main.time_mark >= start) &
    (df_main.time_mark < stop),
    'mark'
] = 'high_fall'

In [13]:
df_main.loc[
    (df_main.sample_num == sample_num) & 
    (df_main.a_s_num == a_s_num) &
    (df_main.time_mark >= start) &
    (df_main.time_mark < stop),
    'mark'
]

190861    high_fall
190862    high_fall
190863    high_fall
190864    high_fall
190865    high_fall
            ...    
195927    high_fall
195928    high_fall
195929    high_fall
195930    high_fall
195931    high_fall
Name: mark, Length: 5071, dtype: category
Categories (26, object): ['low_fall', 'medium_fall', 'high_fall', 'low_rise', ..., 'short_pause', 'medium_pause', 'long_pause', 'plain']

In [14]:
df_main.mark.unique()

['plain', 'high_pre_head', 'stressed_syllable', 'high_fall']
Categories (26, object): ['low_fall', 'medium_fall', 'high_fall', 'low_rise', ..., 'short_pause', 'medium_pause', 'long_pause', 'plain']

In [15]:
df_main.to_parquet('audiosets_main_20_04.parquet')